# 05 · Hands for J.A.R.V.I.S. – tools

A model can only produce text. **Tools** let an agent *do* things: run code, search files, search the web or call your own functions. The model decides **when** to call a tool; something else **executes** it.

| 🛠️ Tool | 🏃 Who executes it | 🧪 Example here |
|---|---|---|
| **Function** | your code (you run it and send the result back) | arc reactor output |
| **Code interpreter** | Foundry sandbox | math and charts |
| **File search** | Foundry (vector store) | the workshop handbook |
| **Web search** | Foundry | current facts |
| **MCP / A2A** | a remote server / agent | chapters 06 and 16 |

Deep dive: [docs/tools.md](../docs/tools.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

import json
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, CodeInterpreterTool, FileSearchTool

oai = openai_client()
created_agents = []
def make_agent(name, instructions, tools):
    a = project().agents.create_version(agent_name=name, definition=PromptAgentDefinition(model=model("chat"), instructions=instructions, tools=tools))
    created_agents.append(name)
    return a

## 1 · Function calling – you run the code
1. Describe the function with a JSON schema.
2. The model answers with a `function_call` item (name + arguments) instead of text.
3. **You** run it and send back a `function_call_output`. The model then writes the final answer.

In [ ]:
def get_arc_reactor_output(suit: str) -> dict:
    data = {"mark 3": 3.0, "mark 42": 8.5, "mark 85": 42.0}
    return {"suit": suit, "output_gw": data.get(suit.lower(), None)}

REGISTRY = {"get_arc_reactor_output": get_arc_reactor_output}

fn_tool = FunctionTool(
    name="get_arc_reactor_output",
    description="Get the arc reactor power output in gigawatts for an Iron Man suit.",
    parameters={"type": "object", "properties": {"suit": {"type": "string", "description": "e.g. 'mark 42'"}},
                "required": ["suit"], "additionalProperties": False},
    strict=True,
)
agent = make_agent("jarvis-nb05-fn", "You are J.A.R.V.I.S. Use tools for facts. Be brief.", [fn_tool])

conv = oai.conversations.create()
resp = oai.responses.create(conversation=conv.id, input="Compare the reactor output of the Mark 3 and Mark 42.", extra_body=agent_ref(agent))

for _ in range(5):          # cap the rounds: a confused model could loop forever
    calls = [i for i in resp.output if i.type == "function_call"]
    if not calls:
        break
    outputs = []
    for c in calls:
        args = json.loads(c.arguments)
        result = REGISTRY[c.name](**args)
        print(f"  tool call: {c.name}({args}) -> {result}")
        outputs.append({"type": "function_call_output", "call_id": c.call_id, "output": json.dumps(result)})
    resp = oai.responses.create(conversation=conv.id, input=outputs, extra_body=agent_ref(agent))

print(resp.output_text)

> 💡 Note the loop: the model may call several functions, or call again after seeing results. Always loop until there is no `function_call` left, and cap the rounds. From chapter 13 on, `run_functions(...)` in [stark_helpers.py](stark_helpers.py) runs this same loop for you.

## 2 · Code interpreter – a sandbox that runs Python
No code from you: the model writes Python, Foundry runs it in a sandbox and returns the result (and files).

In [ ]:
ci = make_agent("jarvis-nb05-ci", "You are J.A.R.V.I.S. For any calculation, write and run Python.", [CodeInterpreterTool()])
r = oai.responses.create(input="How many seconds are in 3.7 years (365.25 days)? Show the exact number.", extra_body=agent_ref(ci))
print(r.output_text)
print("items:", tool_calls(r))
for item in r.output:
    if item.type == "code_interpreter_call":
        print("\n--- code the model wrote ---\n" + (item.code or ""))

## 3 · File search – ask your own documents
Upload a file to a **vector store**; the agent retrieves matching passages and answers from them (RAG without building a pipeline).

In [ ]:
vs = oai.vector_stores.create(name="stark-handbook-nb05")
with open(ROOT / "data" / "workshop-handbook.md", "rb") as f:
    oai.vector_stores.files.upload_and_poll(vector_store_id=vs.id, file=f)
print("vector store:", vs.id)

fs = make_agent("jarvis-nb05-fs", "Answer only from the handbook. If it is not there, say so.", [FileSearchTool(vector_store_ids=[vs.id])])
for q in ["Who approves purchases above 5,000 USD?", "What is the Wi-Fi password?"]:
    r = oai.responses.create(input=q, extra_body=agent_ref(fs))
    print(f"> {q}\n{r.output_text}  {tool_calls(r)}\n")

## 4 · Web search
> ⚠️ Web search sends the query to Bing outside the Azure compliance boundary. Read the data-handling terms before using it with customer data. May not be available in every region.

In [ ]:
from azure.ai.projects.models import WebSearchPreviewTool
try:
    ws = make_agent("jarvis-nb05-ws", "Search the web for current facts and cite sources.", [WebSearchPreviewTool()])
    r = oai.responses.create(input="What is the latest Microsoft Foundry announcement?", extra_body=agent_ref(ws))
    print(r.output_text[:600], tool_calls(r))
except Exception as exc:
    print("web search not available here:", type(exc).__name__, str(exc)[:200])

## 👀 In the portal
**Build → Agents → jarvis-nb05-fs**: the agent with the file search tool and its vector store `stark-handbook-nb05`. Open the other `jarvis-nb05-*` agents to compare their tools. Try the **Playground**.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
oai.conversations.delete(conv.id)
oai.vector_stores.delete(vs.id)
for name in created_agents:
    delete_agent(name)

## 🎯 Recap
Pick the tool by **who should run it**: your code (function), Foundry (code interpreter, file search, web search) or someone else's server (MCP, A2A).

> 💡 Next: [06 · MCP](06_mcp_tools.ipynb) – tools you don't have to write.